# NLP + Machine Learning Project Workflow

Business Understanding
        ↓
Data Collection
        ↓
Dataset Understanding
        ↓
Data Cleaning
        ↓
Exploratory Data Analysis (EDA)
        ↓
Text Preprocessing
        ↓
Train / Validation / Test Split
        ↓
Feature Extraction
(Bag of Words / TF-IDF)
        ↓
Create Training Data
        ↓
Choose ML Model
        ↓
Choose Evaluation Metrics
        ↓
Training
        ↓
Validation
        ↓
Hyperparameter Tuning (Optional)
        ↓
Model Evaluation
        ↓
Error Analysis
        ↓
Save Trained Model
        ↓
Inference / Prediction
        ↓
Deployment
(Streamlit / API)
        ↓
GitHub + README

#### <mark>Project Goal

Build a **Customer Support Ticket Classification** system that predicts, for every incoming support ticket, **two target variables at once**:

* 🏷️ **Category** — the broad department the ticket belongs to (e.g. ORDER, REFUND, ACCOUNT, PAYMENT...)
* 🎯 **Intent** — the exact reason the customer is reaching out (e.g. cancel_order, track_refund, edit_account...)

**Goal:** Automatically read a customer's message and understand *what* it is about and *why* the customer is writing in, without a human having to read it first.

**End Result:**

`Support Ticket → NLP Processing → ML Model → Category + Intent`

**Use:** Auto-route tickets to the right team (category) and trigger the right response workflow (intent) — cutting down manual triage time for the support desk.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv(r"C:\Users\rohit\OneDrive\Desktop\Customer Support Ticket Classification System\data\customer_dataset_raw.csv")

pd.set_option("display.max_colwidth", 60)

df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding canceling ...
1,BQZ,i have a question about cancelling oorder {{Order Number}},ORDER,cancel_order,I've been informed that you have a question about cancel...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance with cancelin...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with canceling you...
4,BCELN,"I cannot afford this order, cancel purchase {{Order Numb...",ORDER,cancel_order,I'm sensitive to the fact that you're facing financial d...


In [3]:
df.columns

Index(['flags', 'instruction', 'category', 'intent', 'response'], dtype='str')

In [4]:
#dropping columns that we don't need for classification
#'flags' is just an internal tagging code and 'response' is the agent's reply, not the customer's message

df.drop(columns = ['flags', 'response'], inplace = True)

# Data Cleaning

In [5]:
df.isnull().sum()

instruction    0
category       0
intent         0
dtype: int64

In [6]:
#this time we have TWO target variables instead of one, so we check both

df['category'].nunique()

11

In [7]:
df['intent'].nunique()

27

In [8]:
df['category'].value_counts()

category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64

In [9]:
df['intent'].value_counts()

intent
check_invoice               1000
complaint                   1000
contact_customer_service    1000
edit_account                1000
switch_account              1000
check_payment_methods        999
contact_human_agent          999
delivery_period              999
get_invoice                  999
newsletter_subscription      999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
set_up_shipping_address      997
delete_account               995
delivery_options             995
recover_password             995
track_order                  995
change_shipping_address      973
check_cancellation_fee       950
Name: count, dtype: int64

In [10]:
df['category'].unique()

<StringArray>
[       'ORDER',     'SHIPPING',       'CANCEL',      'INVOICE',      'PAYMENT',       'REFUND',     'FEEDBACK',
      'CONTACT',      'ACCOUNT',     'DELIVERY', 'SUBSCRIPTION']
Length: 11, dtype: str

In [11]:
df['intent'].unique()

<StringArray>
[            'cancel_order',             'change_order',  'change_shipping_address',   'check_cancellation_fee',
            'check_invoice',    'check_payment_methods',      'check_refund_policy',                'complaint',
 'contact_customer_service',      'contact_human_agent',           'create_account',           'delete_account',
         'delivery_options',          'delivery_period',             'edit_account',              'get_invoice',
               'get_refund',  'newsletter_subscription',            'payment_issue',              'place_order',
         'recover_password',    'registration_problems',                   'review',  'set_up_shipping_address',
           'switch_account',              'track_order',             'track_refund']
Length: 27, dtype: str

In [12]:
#providing indexes to all category labels

categories = list(df['category'].unique())

categories

['ORDER', 'SHIPPING', 'CANCEL', 'INVOICE', 'PAYMENT', 'REFUND', 'FEEDBACK', 'CONTACT', 'ACCOUNT', 'DELIVERY', 'SUBSCRIPTION']

In [13]:
category_number = {}

i = 0

for category in categories:
    category_number[category] = i
    i = i + 1

In [14]:
category_number

{'ORDER': 0, 'SHIPPING': 1, 'CANCEL': 2, 'INVOICE': 3, 'PAYMENT': 4, 'REFUND': 5, 'FEEDBACK': 6, 'CONTACT': 7, 'ACCOUNT': 8, 'DELIVERY': 9, 'SUBSCRIPTION': 10}

In [15]:
#now apply these changes into the original df

df['category'] = df['category'].map(category_number)

In [16]:
#now doing the exact same thing for the 'intent' column

intents = list(df['intent'].unique())

intents

['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoice', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']

In [17]:
intent_number = {}

i = 0

for intent in intents:
    intent_number[intent] = i
    i = i + 1

In [18]:
intent_number

{'cancel_order': 0, 'change_order': 1, 'change_shipping_address': 2, 'check_cancellation_fee': 3, 'check_invoice': 4, 'check_payment_methods': 5, 'check_refund_policy': 6, 'complaint': 7, 'contact_customer_service': 8, 'contact_human_agent': 9, 'create_account': 10, 'delete_account': 11, 'delivery_options': 12, 'delivery_period': 13, 'edit_account': 14, 'get_invoice': 15, 'get_refund': 16, 'newsletter_subscription': 17, 'payment_issue': 18, 'place_order': 19, 'recover_password': 20, 'registration_problems': 21, 'review': 22, 'set_up_shipping_address': 23, 'switch_account': 24, 'track_order': 25, 'track_refund': 26}

In [19]:
#now apply these changes into the original df

df['intent'] = df['intent'].map(intent_number)

In [20]:
#lets verify both columns got mapped correctly

df.sample(3)

                                            instruction  category  intent
4476                      how can I check bill  #12588?         3       4
676         I'm rtying to cancel order {{Order Number}}         0       0
16877  I do not know what to do to demand my money back         5      16

In [21]:
#now labels work is done we will start working on the 'instruction' text as well


#first of all converting everything into lowercase 

df['instruction'] = df['instruction'].apply(lambda x : x.lower())

In [22]:
df.sample(3)

                                              instruction  category  intent
12487  can i place a damn order from {{delivery country}}         9      12
3035    i do not know how i can see the cancellation fees         2       3
11704          i want help canceling the standard account         8      11

#### Next Step in Data Cleaning : Remove punctuation

In [23]:
# step 2 : clean punctuation marks

punctuation_marks = [
    ".",
    ",",
    "?",
    "!",
    ";",
    ":",
    "-",
    "—",
    "–",
    "'",
    '"',
    "(",
    ")",
    "[",
    "]",
    "{",
    "}",
    "...",
    "/",
    "\\"
]


print(*punctuation_marks)

. , ? ! ; : - — – ' " ( ) [ ] { } ... / \


In [24]:
import string
import builtins

def remove_punc(txt):
    return txt.translate(builtins.str.maketrans('', '', string.punctuation))

df['instruction'] = df['instruction'].apply(remove_punc)

# maketrans() creates a translation table for str.translate().
# str.maketrans(from, to, delete)

In [25]:
# 3rd step would be to remove numbers, but the ticket text keeps things like {{Order Number}} and
# ids/quantities that can actually matter for intent (e.g. 'cancel order 12345'), so skipping this

In [26]:
#Remove URL's and link (same as before) — this dataset doesn't really have any tweet-style links,
#but keeping the step so the pipeline stays consistent

def remove_link(txt):
    return txt.split('https')[0].strip()

df['instruction'] = df['instruction'].apply(remove_link)

In [27]:
df.sample(5)

                                                      instruction  ...  intent
20350  i want to reset the bloody key of my user will you help me  ...      20
2430        i want to know more about editing my shipping address  ...       2
7834       how can i lodge a consumer claim against your business  ...       7
22009                   where do i leave a review about a service  ...      22
19983                          i have got to reset my account pwd  ...      20

[5 rows x 3 columns]

In [28]:
#link/html removal done, no emojis or special characters in this dataset either, so skipping those steps

#### Next Step in Data Cleaning : Remove Stopwords

In [29]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [30]:
negation_words = {'no', 'not', 'nor', 'never'}

stop_words = stop_words - negation_words


def remove_stopwords(txt):
    return ' '.join(
        word for word in txt.split()
        if word not in stop_words
    )

df['instruction'] = df['instruction'].apply(remove_stopwords)

# split → check → keep → join

In [31]:
df.sample(5)

                                      instruction  category  intent
10265               open account category account         8      10
23002           cannot set shipping address valid         1      23
1772                 swap item order order number         0       1
22674           im trying write review ur company         6      22
315    want help cancelling purchase order number         0       0

stop words removed now lets continue the project now 

<b>Next Step : Train Test Split

In [32]:
#saving this prepared dataset

df_prepared = df.copy()

df_prepared.to_csv(r"C:\Users\rohit\Downloads\AI and Data Science\Customer Support Ticket Classification System\data\prepared_dataset.csv",
                   index=False)

In [33]:
#this time we have TWO targets, so we keep 'instruction' as X and pull both
#'category' and 'intent' out as separate y's

X = df[['instruction']]
y_category = df['category']
y_intent = df['intent']

In [34]:
from sklearn.model_selection import train_test_split

#splitting once so both targets share the exact same train/test rows

X_train, X_test, y_cat_train, y_cat_test, y_int_train, y_int_test = train_test_split(
    X, y_category, y_intent, test_size=0.2, random_state=42
)

## Train Test Split done now next step is Feature Extraction / Vectorization

Goal : Convert words into numbers

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer()

X_train_bow = bow_vectorizer.fit_transform(X_train['instruction'])
X_test_bow = bow_vectorizer.transform(X_test['instruction'])

In [36]:
X_train_bow.shape
X_test_bow.shape

(5375, 2448)

In [37]:
X_train_bow

<21497x2448 sparse matrix of type '<class 'numpy.int64'>'
	with 156117 stored elements in Compressed Sparse Row format>

In [38]:
#same for tfidf

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train['instruction'])
X_test_tfidf = tfidf_vectorizer.transform(X_test['instruction'])

In [39]:
X_train_tfidf.shape
X_test_tfidf.shape

(5375, 2448)

## Feature Extraction / Vectorization done now next step is Model Training

<b>Use of models like:

- Naive Bayes
- Logistic Regression
- SVM (Support Vector Machines)

Since we now have **two target variables (category & intent)**, we train the same models twice — once per target — but reuse the same `X_train_bow` / `X_train_tfidf` features for both.

### 🏷️ Category Model Training

In [40]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


nb_model_cat_bow = MultinomialNB()

nb_model_cat_bow.fit(X_train_bow, y_cat_train)

cat_pred_bow = nb_model_cat_bow.predict(X_test_bow)
print("Category Bow Model Accuracy :", accuracy_score(y_cat_test, cat_pred_bow))

Category Bow Model Accuracy : 0.9973953488372093


In [41]:
nb_model_cat_tfidf = MultinomialNB()

nb_model_cat_tfidf.fit(X_train_tfidf, y_cat_train)
cat_pred_tfidf = nb_model_cat_tfidf.predict(X_test_tfidf)

print("Category TF-IDF Model Accuracy :", accuracy_score(y_cat_test, cat_pred_tfidf))

Category TF-IDF Model Accuracy : 0.9979534883720931


| Feature Extraction | Model                   |   Accuracy |
| ------------------ | ----------------------- | ---------: |
| **BOW**            | Multinomial Naive Bayes | **99.74%** |
| **TF-IDF**         | Multinomial Naive Bayes | **99.80%** |


In [42]:
#now we train Logistic Regression for category, same as we trained MultinomialNB


#using bow with logistic

from sklearn.linear_model import LogisticRegression

logistic_model_cat_bow = LogisticRegression(max_iter=1000)

logistic_model_cat_bow.fit(X_train_bow, y_cat_train)

log_cat_pred_bow = logistic_model_cat_bow.predict(X_test_bow)

print("Category Logistic Regression + BOW Accuracy :", accuracy_score(y_cat_test, log_cat_pred_bow))

Category Logistic Regression + BOW Accuracy : 0.9977674418604651


In [43]:
#now logistic with tf-idf, for category

from sklearn.linear_model import LogisticRegression

logistic_model_cat_tfidf = LogisticRegression(max_iter=1000)

logistic_model_cat_tfidf.fit(X_train_tfidf, y_cat_train)

log_cat_pred_tfidf = logistic_model_cat_tfidf.predict(X_test_tfidf)
print("Category Logistic Regression + TF-IDF Accuracy :", accuracy_score(y_cat_test, log_cat_pred_tfidf))

Category Logistic Regression + TF-IDF Accuracy : 0.9975813953488372


| Model                       |           BOW |     TF-IDF |
| --------------------------- | ------------: | ---------: |
| **Multinomial Naive Bayes** |    **99.74%** | **99.80%** |
| **Logistic Regression**     | **99.78%** 🏆 | **99.76%** |


### 🎯 Intent Model Training

In [44]:
nb_model_int_bow = MultinomialNB()

nb_model_int_bow.fit(X_train_bow, y_int_train)

int_pred_bow = nb_model_int_bow.predict(X_test_bow)
print("Intent Bow Model Accuracy :", accuracy_score(y_int_test, int_pred_bow))

Intent Bow Model Accuracy : 0.9864186046511628


In [45]:
nb_model_int_tfidf = MultinomialNB()

nb_model_int_tfidf.fit(X_train_tfidf, y_int_train)
int_pred_tfidf = nb_model_int_tfidf.predict(X_test_tfidf)

print("Intent TF-IDF Model Accuracy :", accuracy_score(y_int_test, int_pred_tfidf))

Intent TF-IDF Model Accuracy : 0.9890232558139535


| Feature Extraction | Model                   |   Accuracy |
| ------------------ | ----------------------- | ---------: |
| **BOW**            | Multinomial Naive Bayes | **98.64%** |
| **TF-IDF**         | Multinomial Naive Bayes | **98.90%** |


In [46]:
#now Logistic Regression for intent

logistic_model_int_bow = LogisticRegression(max_iter=1000)

logistic_model_int_bow.fit(X_train_bow, y_int_train)

log_int_pred_bow = logistic_model_int_bow.predict(X_test_bow)

print("Intent Logistic Regression + BOW Accuracy :", accuracy_score(y_int_test, log_int_pred_bow))

Intent Logistic Regression + BOW Accuracy : 0.9916279069767442


In [47]:
#now logistic with tf-idf, for intent

logistic_model_int_tfidf = LogisticRegression(max_iter=1000)

logistic_model_int_tfidf.fit(X_train_tfidf, y_int_train)

log_int_pred_tfidf = logistic_model_int_tfidf.predict(X_test_tfidf)
print("Intent Logistic Regression + TF-IDF Accuracy :", accuracy_score(y_int_test, log_int_pred_tfidf))

Intent Logistic Regression + TF-IDF Accuracy : 0.990139534883721


| Model                       |           BOW |     TF-IDF |
| --------------------------- | ------------: | ---------: |
| **Multinomial Naive Bayes** |    **98.64%** | **98.90%** |
| **Logistic Regression**     | **99.16%** 🏆 | **99.01%** |


Logistic Regression + BOW comes out on top for both targets, same as it did in the sentiment analysis project — so that's what we'll save as the final model for each target.

In [48]:
# Saving the trained category model and vectorizer

import os
import joblib

model_path = r"C:\Users\rohit\Downloads\AI and Data Science\Customer Support Ticket Classification System\models"

# Create models folder if it doesn't exist
os.makedirs(model_path, exist_ok=True)

# Save category model
joblib.dump(logistic_model_cat_bow, os.path.join(model_path, "logistic_model_category_bow.pkl"))

# Save BOW vectorizer (shared by both targets)
joblib.dump(bow_vectorizer, os.path.join(model_path, "bow_vectorizer.pkl"))

print("Category model and vectorizer saved successfully!")
print("Saved at:", model_path)

Category model and vectorizer saved successfully!
Saved at: C:\Users\rohit\Downloads\AI and Data Science\Customer Support Ticket Classification System\models


In [49]:
# Saving the trained intent model too

joblib.dump(logistic_model_int_bow, os.path.join(model_path, "logistic_model_intent_bow.pkl"))

print("Intent model saved successfully!")
print("Saved at:", model_path)

Intent model saved successfully!
Saved at: C:\Users\rohit\Downloads\AI and Data Science\Customer Support Ticket Classification System\models


In [50]:
# ============================================================
# EXTRA SAVES — run this after your existing model/vectorizer save cell
# ============================================================
import os
import joblib
import json
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

model_path = r"C:\Users\rohit\Downloads\AI and Data Science\Customer Support Ticket Classification System\models"
os.makedirs(model_path, exist_ok=True)

# 1. Label mappings (needed to decode predictions back to category/intent names)
category_number_inv = {v: k for k, v in category_number.items()}
intent_number_inv = {v: k for k, v in intent_number.items()}

with open(os.path.join(model_path, "category_label_mapping.json"), "w") as f:
    json.dump(category_number_inv, f)

with open(os.path.join(model_path, "intent_label_mapping.json"), "w") as f:
    json.dump(intent_number_inv, f)

# 2. Evaluation report + confusion matrix for CATEGORY (for portfolio / README)
cat_report = classification_report(y_cat_test, log_cat_pred_bow, target_names=categories, output_dict=True)
cat_cm = confusion_matrix(y_cat_test, log_cat_pred_bow)

with open(os.path.join(model_path, "category_eval_report.json"), "w") as f:
    json.dump(cat_report, f, indent=2)

np.save(os.path.join(model_path, "category_confusion_matrix.npy"), cat_cm)

# 3. Evaluation report + confusion matrix for INTENT
int_report = classification_report(y_int_test, log_int_pred_bow, target_names=intents, output_dict=True)
int_cm = confusion_matrix(y_int_test, log_int_pred_bow)

with open(os.path.join(model_path, "intent_eval_report.json"), "w") as f:
    json.dump(int_report, f, indent=2)

np.save(os.path.join(model_path, "intent_confusion_matrix.npy"), int_cm)

print("Category Classification Report:\n", classification_report(y_cat_test, log_cat_pred_bow, target_names=categories))
print("Category Confusion Matrix:\n", cat_cm)

print("\nIntent Classification Report:\n", classification_report(y_int_test, log_int_pred_bow, target_names=intents))
print("Intent Confusion Matrix:\n", int_cm)

print("\nAll extra artifacts saved:")
print("- category_label_mapping.json + intent_label_mapping.json")
print("- category_eval_report.json + category_confusion_matrix.npy")
print("- intent_eval_report.json + intent_confusion_matrix.npy")

Category Classification Report:
               precision    recall  f1-score   support

       ORDER       1.00      1.00      1.00       763
    SHIPPING       1.00      0.99      1.00       444
      CANCEL       1.00      1.00      1.00       199
     INVOICE       1.00      1.00      1.00       407
     PAYMENT       1.00      0.99      0.99       410
      REFUND       1.00      1.00      1.00       601
    FEEDBACK       1.00      1.00      1.00       427
     CONTACT       1.00      1.00      1.00       409
     ACCOUNT       0.99      1.00      1.00      1160
    DELIVERY       0.99      1.00      1.00       389
SUBSCRIPTION       1.00      0.99      1.00       166

    accuracy                           1.00      5375
   macro avg       1.00      1.00      1.00      5375
weighted avg       1.00      1.00      1.00      5375

Category Confusion Matrix:
 array([[ 762,    0,    0,    0,    0,    0,    0,    0,    1,    0,    0],
       [   1,  441,    0,    0,    0,    0,    0,  

<div style="
background:linear-gradient(90deg,#D5F5E3,#E8F8F5);
border-left:6px solid #27AE60;
padding:16px 22px;
margin:15px 0;
border-radius:6px;
">

<h2 style="
color:#1E8449;
font-family:sans-serif;
font-weight:800;
margin:0;
letter-spacing:1px;
">
COMPLETED
</h2>

</div>